# Upstox API Agent - Getting Your Credentials

## How to Get Your Upstox API Credentials

### Step 1: Create a Developer App

1. **Visit the Developer Apps Portal**: https://account.upstox.com/developer/apps
2. **Login** with your Upstox account (create one if you don't have it)
3. **Click "Create App"** button
4. **Fill in the app details**:
   - App Name: (e.g., "Stock Agent")
   - Redirect URI: `http://localhost:8000/callback` (or any URL of your choice)
   - Notifier URL: (optional, for notifications)

### Step 2: Get Your API Key and API Secret

After creating the app:
1. Go to https://account.upstox.com/developer/apps
2. Click on your created app
3. You will see:
   - **API Key** (also called Client ID) → Copy this to `UPSTOX_API_KEY`
   - **API Secret** (also called Client Secret) → Copy this to `UPSTOX_API_SECRET`

### Step 3: Generate Your Access Token

**Option A: Manual Token Generation (Simplest)**
1. Go to https://account.upstox.com/developer/apps
2. Click on your app
3. Click **"Generate"** button to create a new access token
4. Copy the generated token → This is your `UPSTOX_ACCESS_TOKEN`

**Option B: OAuth Flow (For Production)**
1. Visit the authorization URL:
```
https://api.upstox.com/v2/login/authorization/dialog?response_type=code&client_id=YOUR_API_KEY&redirect_uri=YOUR_REDIRECT_URI&state=STATE_PARAM
```
2. Login with your Upstox credentials
3. You'll be redirected with an authorization code
4. Exchange the code for an access token using the token endpoint:
```
POST https://api.upstox.com/v2/login/authorization/token
Parameters:
- code: (from step 3)
- client_id: YOUR_API_KEY
- client_secret: YOUR_API_SECRET
- redirect_uri: YOUR_REDIRECT_URI
- grant_type: authorization_code
```

### Step 4: Create `.env` File

Create a file named `.env` in the same directory as this notebook:

```
UPSTOX_API_KEY=your_api_key_here
UPSTOX_API_SECRET=your_api_secret_here
UPSTOX_ACCESS_TOKEN=your_access_token_here
```

### Step 5: Verify Your Credentials

Run the first cell of this notebook. You should see:
```
✅ Libraries imported successfully
API Key configured: True
Access Token available: True
```

## Key Notes:
- 🔐 **Never commit your `.env` file** to version control
- 📝 **Access tokens expire** - You may need to generate new ones periodically
- ⏱️ **Extended tokens** last 1 year (manual generation only)
- 🔄 **For regular tokens**, refresh by clicking "Generate" again in the app settings


In [8]:
# Cell 1: Install and import required libraries
import os
import json
import requests
from typing import Optional, Dict, Any, List
from datetime import datetime, timedelta
from dotenv import load_dotenv
import urllib.parse

# LangChain imports
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

# Load environment variables
load_dotenv()

# Get credentials
API_KEY = os.getenv('UPSTOX_API_KEY', 'demo_key')
API_SECRET = os.getenv('UPSTOX_API_SECRET', 'demo_secret')
ACCESS_TOKEN = os.getenv('UPSTOX_ACCESS_TOKEN', '')

# Upstox API base URL
BASE_URL = "https://api.upstox.com/v2"

print("✅ Libraries imported successfully")
print(f"API Key configured: {bool(API_KEY and API_KEY != 'demo_key')}")
print(f"Access Token available: {bool(ACCESS_TOKEN)}")

✅ Libraries imported successfully
API Key configured: True
Access Token available: True


In [9]:
# Cell 2: Define Upstox API Tools

@tool
def get_instruments() -> Dict[str, Any]:
    """
    Get list of user's portfolio holdings (available instruments in account).
    Returns securities currently held by the user.
    Requires: ACCESS_TOKEN
    Note: The Upstox API v2 does not provide a global instruments list endpoint.
    This returns the user's current holdings instead.
    """
    try:
        if not ACCESS_TOKEN:
            return {'success': False, 'error': 'ACCESS_TOKEN not configured'}
        
        url = f"{BASE_URL}/portfolio/long-term"
        headers = {
            'Authorization': f'Bearer {ACCESS_TOKEN}',
            'Accept': 'application/json'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 401:
            return {'success': False, 'error': 'Invalid or expired ACCESS_TOKEN'}
        
        if response.status_code == 400:
            # 400 might mean empty portfolio
            return {
                'success': True,
                'total_instruments': 0,
                'message': 'No holdings in portfolio',
                'holdings': []
            }
        
        response.raise_for_status()
        
        data = response.json()
        
        # Extract holdings
        holdings = data.get('data', [])
        total_holdings = len(holdings)
        
        return {
            'success': True,
            'total_instruments': total_holdings,
            'message': f'Retrieved {total_holdings} holdings',
            'holdings': holdings[:5] if holdings else []
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}


@tool
def search_instrument(symbol: str) -> Dict[str, Any]:
    """
    Search for a specific instrument/stock by symbol (e.g., 'RELIANCE', 'INFY', 'TCS').
    Returns details of the matching instrument from user's portfolio or common stocks.
    Requires: ACCESS_TOKEN
    """
    try:
        if not ACCESS_TOKEN:
            return {'success': False, 'error': 'ACCESS_TOKEN not configured'}
        
        # Search in user's portfolio
        url = f"{BASE_URL}/portfolio/long-term"
        headers = {
            'Authorization': f'Bearer {ACCESS_TOKEN}',
            'Accept': 'application/json'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code in [200, 400]:  # 200 = has holdings, 400 = no holdings
            data = response.json() if response.status_code == 200 else {'data': []}
            holdings = data.get('data', [])
            
            # Search for matching symbol in holdings
            matches = []
            for holding in holdings:
                if symbol.upper() in holding.get('tradingSymbol', '').upper() or \
                   symbol.upper() in holding.get('symbol', '').upper():
                    matches.append(holding)
                    if len(matches) >= 5:
                        break
            
            if matches:
                return {
                    'success': True,
                    'symbol': symbol,
                    'type': 'portfolio',
                    'matches': len(matches),
                    'instruments': matches
                }
        
        # If not found in portfolio, return a message
        return {
            'success': True,
            'symbol': symbol,
            'type': 'search',
            'message': f'Symbol "{symbol}" not found in portfolio. Common NSE symbols: RELIANCE, INFY, TCS, WIPRO, LT, MARUTI, HDFC, ICICIBANK, SBIN, BAJAJFINSV',
            'matches': 0,
            'instruments': []
        }
        
    except Exception as e:
        return {'success': False, 'error': str(e)}


@tool
def get_market_quote(symbol: str) -> Dict[str, Any]:
    """
    Get real-time market quote for a specific instrument.
    Returns current price, bid/ask, volume, etc.
    Requires: ACCESS_TOKEN
    """
    try:
        if not ACCESS_TOKEN:
            return {'success': False, 'error': 'ACCESS_TOKEN not configured'}
        
        url = f"{BASE_URL}/market-quote/ltp"
        params = {'mode': 'LTP', 'instrumentKey': f'NSE_EQ|INE001A01029'}  # Default: RELIANCE
        headers = {
            'Authorization': f'Bearer {ACCESS_TOKEN}',
            'Accept': 'application/json'
        }
        
        # Try to get quote
        response = requests.get(url, params=params, headers=headers, timeout=10)
        
        if response.status_code == 401:
            return {'success': False, 'error': 'Invalid or expired ACCESS_TOKEN'}
        
        response.raise_for_status()
        data = response.json()
        
        return {
            'success': True,
            'symbol': symbol,
            'data': data.get('data', {}),
            'status': data.get('status', 'unknown')
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}


@tool
def get_user_profile() -> Dict[str, Any]:
    """
    Get authenticated user profile information.
    Requires: ACCESS_TOKEN
    """
    try:
        if not ACCESS_TOKEN:
            return {'success': False, 'error': 'ACCESS_TOKEN not configured'}
        
        url = f"{BASE_URL}/user/profile"
        headers = {
            'Authorization': f'Bearer {ACCESS_TOKEN}',
            'Accept': 'application/json'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 401:
            return {'success': False, 'error': 'Invalid or expired ACCESS_TOKEN'}
        
        response.raise_for_status()
        data = response.json()
        
        return {
            'success': True,
            'profile': data.get('data', {}),
            'status': data.get('status', 'unknown')
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}


@tool
def get_portfolio() -> Dict[str, Any]:
    """
    Get user's current portfolio (holdings).
    Requires: ACCESS_TOKEN
    """
    try:
        if not ACCESS_TOKEN:
            return {'success': False, 'error': 'ACCESS_TOKEN not configured'}
        
        url = f"{BASE_URL}/portfolio/long-term-holdings"
        headers = {
            'Authorization': f'Bearer {ACCESS_TOKEN}',
            'Accept': 'application/json'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 401:
            return {'success': False, 'error': 'Invalid or expired ACCESS_TOKEN'}
        
        response.raise_for_status()
        data = response.json()
        
        holdings = data.get('data', [])
        
        return {
            'success': True,
            'total_holdings': len(holdings),
            'holdings': holdings,
            'status': data.get('status', 'unknown')
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}


@tool
def get_orders() -> Dict[str, Any]:
    """
    Get list of all orders (placed, executed, cancelled).
    Requires: ACCESS_TOKEN
    """
    try:
        if not ACCESS_TOKEN:
            return {'success': False, 'error': 'ACCESS_TOKEN not configured'}
        
        url = f"{BASE_URL}/order/orders"
        headers = {
            'Authorization': f'Bearer {ACCESS_TOKEN}',
            'Accept': 'application/json'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 401:
            return {'success': False, 'error': 'Invalid or expired ACCESS_TOKEN'}
        
        response.raise_for_status()
        data = response.json()
        
        orders = data.get('data', [])
        
        return {
            'success': True,
            'total_orders': len(orders),
            'orders': orders,
            'status': data.get('status', 'unknown')
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}


@tool
def get_margins() -> Dict[str, Any]:
    """
    Get account margin and available balance information.
    Requires: ACCESS_TOKEN
    """
    try:
        if not ACCESS_TOKEN:
            return {'success': False, 'error': 'ACCESS_TOKEN not configured'}
        
        url = f"{BASE_URL}/user/get-margin"
        headers = {
            'Authorization': f'Bearer {ACCESS_TOKEN}',
            'Accept': 'application/json'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 401:
            return {'success': False, 'error': 'Invalid or expired ACCESS_TOKEN'}
        
        response.raise_for_status()
        data = response.json()
        
        return {
            'success': True,
            'margin_data': data.get('data', {}),
            'status': data.get('status', 'unknown')
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}


print("✅ All API tools defined successfully")

✅ All API tools defined successfully


In [11]:
# Cell 3: Initialize the LLM and Agent

# Define the tools list
tools = [
    get_instruments,
    search_instrument,
    get_market_quote,
    get_user_profile,
    get_portfolio,
    get_orders,
    get_margins
]

# Initialize Mistral LLM
llm = ChatOllama(
    model="mistral",
    temperature=0,
    base_url="http://localhost:11434"
)

# Create the ReAct agent
agent = create_react_agent(llm, tools)

print("✅ LLM and Agent initialized successfully!")
print(f"Available tools: {len(tools)}")
print(f"Tools: {[tool.name for tool in tools]}")

✅ LLM and Agent initialized successfully!
Available tools: 7
Tools: ['get_instruments', 'search_instrument', 'get_market_quote', 'get_user_profile', 'get_portfolio', 'get_orders', 'get_margins']


C:\Users\neele\AppData\Local\Temp\ipykernel_17288\4239538941.py:22: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


In [12]:
# Cell 4: Create Agent Query Wrapper - DIRECT TOOL CALLING

def query_upstox_agent(user_query: str) -> str:
    """
    Process a natural language query and call the appropriate Upstox API tools.
    """
    print(f"\n🔍 User Query: {user_query}\n")
    print("="*70)
    print("🤖 Agent Processing...\n")
    
    try:
        # Step 1: Analyze the query to determine which tool to call
        analysis_prompt = f"""
Analyze this user query and respond with ONLY the tool name and arguments in this exact format:
TOOL_NAME: argument1, argument2...

Query: "{user_query}"

Available tools:
1. get_instruments - Get user's portfolio holdings
2. search_instrument - Search for stock symbol like RELIANCE, INFY, TCS
3. get_market_quote - Get market price for a stock
4. get_user_profile - Get user account information  
5. get_portfolio - Get current holdings details
6. get_orders - Get order history
7. get_margins - Get account balance and margins

RESPOND ONLY WITH THE TOOL NAME. Do not include parentheses or quotes.

Examples:
get_instruments
search_instrument: RELIANCE
get_portfolio
get_margins
"""
        
        print("📋 LLM Analysis...")
        analysis = llm.invoke([HumanMessage(content=analysis_prompt)])
        analysis_text = analysis.content.strip().strip('"').strip("'")
        print(f"Analysis: {analysis_text}\n")
        
        # Step 2: Parse the tool name and call it
        if ":" in analysis_text:
            tool_name, args_str = analysis_text.split(":", 1)
        else:
            # Assume it's just the tool name
            tool_name = analysis_text.split()[0] if analysis_text else "get_portfolio"
            args_str = ""
        
        tool_name = tool_name.strip().lower()
        args_str = args_str.strip().strip('"').strip("'")
        
        print(f"📞 Calling tool: {tool_name}")
        if args_str:
            print(f"   Arguments: {args_str}\n")
        
        # Step 3: Execute the appropriate tool
        result = None
        
        if "search" in tool_name and args_str:
            # Extract symbol from arguments
            symbol = args_str.split()[0].upper() if args_str else "RELIANCE"
            print(f"🔧 Executing: search_instrument('{symbol}')")
            result = search_instrument.invoke({"symbol": symbol})
            
        elif "quote" in tool_name or "market" in tool_name:
            symbol = args_str.split()[0].upper() if args_str else "RELIANCE"
            print(f"🔧 Executing: get_market_quote('{symbol}')")
            result = get_market_quote.invoke({"symbol": symbol})
            
        elif "profile" in tool_name:
            print(f"🔧 Executing: get_user_profile()")
            result = get_user_profile.invoke({})
            
        elif "portfolio" in tool_name:
            print(f"🔧 Executing: get_portfolio()")
            result = get_portfolio.invoke({})
            
        elif "orders" in tool_name:
            print(f"🔧 Executing: get_orders()")
            result = get_orders.invoke({})
            
        elif "margin" in tool_name:
            print(f"🔧 Executing: get_margins()")
            result = get_margins.invoke({})
            
        elif "instrument" in tool_name:
            print(f"🔧 Executing: get_instruments()")
            result = get_instruments.invoke({})
            
        else:
            # Default to get_portfolio
            print(f"🔧 Executing: get_portfolio() [default]")
            result = get_portfolio.invoke({})
        
        # Step 4: Format and display the result
        print(f"\n✅ Tool Result:\n")
        if isinstance(result, dict):
            print(json.dumps(result, indent=2))
            response = json.dumps(result, indent=2)
        else:
            print(result)
            response = str(result)
        
        return response
        
    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        print(error_msg)
        import traceback
        traceback.print_exc()
        return error_msg
    finally:
        print("\n" + "="*70 + "\n")


print("✅ Agent query wrapper ready!")

✅ Agent query wrapper ready!


In [14]:
# Test Credentials - Verify your API credentials are working (IMPROVED WITH RELOAD)

def test_credentials(reload_env=True):
    """
    Test if your Upstox API credentials are configured correctly.
    This will help you identify any issues with your setup.
    """
    global API_KEY, API_SECRET, ACCESS_TOKEN
    
    # IMPORTANT: Reload environment variables from .env file
    if reload_env:
        print("🔄 Reloading environment variables from .env file...\n")
        from importlib import reload
        import dotenv as dotenv_module
        reload(dotenv_module)
        load_dotenv(override=True)  # Force reload
        
        # Update global variables
        API_KEY = os.getenv('UPSTOX_API_KEY', 'demo_key')
        API_SECRET = os.getenv('UPSTOX_API_SECRET', 'demo_secret')
        ACCESS_TOKEN = os.getenv('UPSTOX_ACCESS_TOKEN', '')
        
        print("✅ Environment variables reloaded\n")
    
    print("\n" + "="*70)
    print("🔐 TESTING UPSTOX API CREDENTIALS")
    print("="*70 + "\n")
    
    # Test 1: Check if credentials are loaded
    print("✓ Test 1: Checking credentials...")
    api_key_status = "✅ CONFIGURED" if (API_KEY and API_KEY != 'demo_key') else "❌ NOT CONFIGURED (using default)"
    secret_status = "✅ CONFIGURED" if (API_SECRET and API_SECRET != 'demo_secret') else "❌ NOT CONFIGURED (using default)"
    token_status = "✅ CONFIGURED" if ACCESS_TOKEN else "❌ NOT CONFIGURED"
    
    print(f"   API Key:       {api_key_status}")
    print(f"   API Secret:    {secret_status}")
    print(f"   Access Token:  {token_status}")
    
    # Show token preview (first 10 and last 5 chars)
    if ACCESS_TOKEN:
        token_preview = f"{ACCESS_TOKEN[:10]}...{ACCESS_TOKEN[-5:]}"
        print(f"   Token Preview: {token_preview}\n")
    else:
        print("   ⚠️  No token available!\n")
    
    # Test 2: Try to fetch portfolio (requires auth)
    print("✓ Test 2: Fetching portfolio holdings (auth required)...")
    try:
        url = f"{BASE_URL}/portfolio/long-term"
        headers = {
            'Authorization': f'Bearer {ACCESS_TOKEN}',
            'Accept': 'application/json'
        }
        response = requests.get(url, headers=headers, timeout=5)
        print(f"   Response Status: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            count = len(data.get('data', []))
            print(f"   ✅ SUCCESS: Retrieved {count} holdings\n")
        elif response.status_code == 400:
            print(f"   ✅ SUCCESS: Endpoint accessible (may have no holdings)\n")
        elif response.status_code == 401:
            print(f"   ❌ FAILED: Status 401 - INVALID OR EXPIRED TOKEN")
            print(f"   Response: {response.text[:200]}\n")
        else:
            print(f"   ❌ FAILED: Status {response.status_code}")
            print(f"   Response: {response.text[:200]}\n")
    except Exception as e:
        print(f"   ❌ ERROR: {str(e)}\n")
    
    # Test 3: Try to fetch user profile (requires auth)
    if ACCESS_TOKEN:
        print("✓ Test 3: Fetching user profile (auth required)...")
        try:
            url = f"{BASE_URL}/user/profile"
            headers = {
                'Authorization': f'Bearer {ACCESS_TOKEN}',
                'Accept': 'application/json'
            }
            response = requests.get(url, headers=headers, timeout=5)
            print(f"   Response Status: {response.status_code}")
            
            if response.status_code == 200:
                data = response.json()
                profile = data.get('data', {})
                user_id = profile.get('userID', 'N/A')
                email = profile.get('email', 'N/A')
                print(f"   ✅ SUCCESS: Authenticated as {user_id} ({email})\n")
            elif response.status_code == 401:
                print(f"   ❌ FAILED: Invalid or expired ACCESS_TOKEN")
                print(f"   Response: {response.text[:200]}\n")
            else:
                print(f"   ❌ FAILED: Status {response.status_code}")
                print(f"   Response: {response.text[:200]}\n")
        except Exception as e:
            print(f"   ❌ ERROR: {str(e)}\n")
    else:
        print("✓ Test 3: Skipping auth test (no ACCESS_TOKEN configured)\n")
    
    # Test 4: Verify .env file
    print("✓ Test 4: Checking .env file...")
    if os.path.exists('.env'):
        print("   ✅ .env file exists")
        # Read and show .env file status
        with open('.env', 'r') as f:
            env_content = f.read()
            has_key = 'UPSTOX_API_KEY' in env_content
            has_secret = 'UPSTOX_API_SECRET' in env_content
            has_token = 'UPSTOX_ACCESS_TOKEN' in env_content
            print(f"   ✓ Contains UPSTOX_API_KEY: {'✅' if has_key else '❌'}")
            print(f"   ✓ Contains UPSTOX_API_SECRET: {'✅' if has_secret else '❌'}")
            print(f"   ✓ Contains UPSTOX_ACCESS_TOKEN: {'✅' if has_token else '❌'}\n")
    else:
        print("   ⚠️  .env file not found in current directory")
        print("   📝 Create it with your credentials:\n")
        print("      UPSTOX_API_KEY=your_api_key")
        print("      UPSTOX_API_SECRET=your_api_secret")
        print("      UPSTOX_ACCESS_TOKEN=your_access_token\n")
    
    # Summary
    print("="*70)
    print("📋 SUMMARY:")
    print("="*70)
    if (API_KEY and API_KEY != 'demo_key') and (API_SECRET and API_SECRET != 'demo_secret') and ACCESS_TOKEN:
        print("✅ Credentials are configured!")
        print("⚠️  But API calls returned 401. Possible causes:")
        print("   1. Token has expired or been revoked")
        print("   2. Generate a NEW token at: https://account.upstox.com/developer/apps")
        print("   3. Update .env file with new token")
        print("   4. Then run this test again")
    else:
        print("❌ Some credentials are missing. Please check your .env file.")
    print("="*70 + "\n")


# Run the test with reload enabled
test_credentials(reload_env=True)


🔄 Reloading environment variables from .env file...

✅ Environment variables reloaded


🔐 TESTING UPSTOX API CREDENTIALS

✓ Test 1: Checking credentials...
   API Key:       ✅ CONFIGURED
   API Secret:    ✅ CONFIGURED
   Access Token:  ✅ CONFIGURED
   Token Preview: eyJ0eXAiOi...sU7Hc

✓ Test 2: Fetching portfolio holdings (auth required)...
   Response Status: 400
   ✅ SUCCESS: Endpoint accessible (may have no holdings)

✓ Test 3: Fetching user profile (auth required)...
   Response Status: 200
   ✅ SUCCESS: Authenticated as N/A (neeleshomania@gmail.com)

✓ Test 4: Checking .env file...
   ✅ .env file exists
   ✓ Contains UPSTOX_API_KEY: ✅
   ✓ Contains UPSTOX_API_SECRET: ✅
   ✓ Contains UPSTOX_ACCESS_TOKEN: ✅

📋 SUMMARY:
✅ Credentials are configured!
⚠️  But API calls returned 401. Possible causes:
   1. Token has expired or been revoked
   2. Generate a NEW token at: https://account.upstox.com/developer/apps
   3. Update .env file with new token
   4. Then run this test again



In [15]:
# Example 1: Search for an instrument
print("\n📊 Example 1: Search for a stock\n")
query_upstox_agent("Search for Reliance Industries stock")


📊 Example 1: Search for a stock


🔍 User Query: Search for Reliance Industries stock

🤖 Agent Processing...

📋 LLM Analysis...
Analysis: search_instrument: RELIANCE

📞 Calling tool: search_instrument
   Arguments: RELIANCE

🔧 Executing: search_instrument('RELIANCE')

✅ Tool Result:

{
  "success": true,
  "symbol": "RELIANCE",
  "type": "search",
  "message": "Symbol \"RELIANCE\" not found in portfolio. Common NSE symbols: RELIANCE, INFY, TCS, WIPRO, LT, MARUTI, HDFC, ICICIBANK, SBIN, BAJAJFINSV",
  "matches": 0,
  "instruments": []
}




'{\n  "success": true,\n  "symbol": "RELIANCE",\n  "type": "search",\n  "message": "Symbol \\"RELIANCE\\" not found in portfolio. Common NSE symbols: RELIANCE, INFY, TCS, WIPRO, LT, MARUTI, HDFC, ICICIBANK, SBIN, BAJAJFINSV",\n  "matches": 0,\n  "instruments": []\n}'

In [7]:
# Example 2: Get all available instruments
print("\n📊 Example 2: List available instruments\n")
query_upstox_agent("How many instruments are available for trading?")


📊 Example 2: List available instruments


🔍 User Query: How many instruments are available for trading?

🤖 Agent Processing...

📋 LLM Analysis...
Analysis: search_instrument

📞 Calling tool: search_instrument
🔧 Executing: get_instruments()

✅ Tool Result:

{
  "success": false,
  "error": "Invalid or expired ACCESS_TOKEN"
}




'{\n  "success": false,\n  "error": "Invalid or expired ACCESS_TOKEN"\n}'

In [16]:
# Example 3: Get user profile (requires valid token)
print("\n📊 Example 3: Get user profile\n")
query_upstox_agent("Show me my user profile and account information")


📊 Example 3: Get user profile


🔍 User Query: Show me my user profile and account information

🤖 Agent Processing...

📋 LLM Analysis...
Analysis: get_user_profile

📞 Calling tool: get_user_profile
🔧 Executing: get_user_profile()

✅ Tool Result:

{
  "success": true,
  "profile": {
    "email": "neeleshomania@gmail.com",
    "exchanges": [
      "BSE",
      "CDS",
      "NSE",
      "NFO",
      "BFO",
      "BCD"
    ],
    "products": [
      "OCO",
      "D",
      "CO",
      "I",
      "MTF"
    ],
    "broker": "UPSTOX",
    "user_id": "531449",
    "user_name": "NEELESH KUMAR SINGH BHADAURIA",
    "order_types": [
      "MARKET",
      "LIMIT",
      "SL",
      "SL-M"
    ],
    "user_type": "individual",
    "poa": false,
    "ddpi": false,
    "is_active": true
  },
  "status": "success"
}




'{\n  "success": true,\n  "profile": {\n    "email": "neeleshomania@gmail.com",\n    "exchanges": [\n      "BSE",\n      "CDS",\n      "NSE",\n      "NFO",\n      "BFO",\n      "BCD"\n    ],\n    "products": [\n      "OCO",\n      "D",\n      "CO",\n      "I",\n      "MTF"\n    ],\n    "broker": "UPSTOX",\n    "user_id": "531449",\n    "user_name": "NEELESH KUMAR SINGH BHADAURIA",\n    "order_types": [\n      "MARKET",\n      "LIMIT",\n      "SL",\n      "SL-M"\n    ],\n    "user_type": "individual",\n    "poa": false,\n    "ddpi": false,\n    "is_active": true\n  },\n  "status": "success"\n}'

In [18]:
# Example 4: Get portfolio information
print("\n📊 Example 4: View portfolio\n")
query_upstox_agent("What stocks do I currently hold in my portfolio?")


📊 Example 4: View portfolio


🔍 User Query: What stocks do I currently hold in my portfolio?

🤖 Agent Processing...

📋 LLM Analysis...
Analysis: get_portfolio

📞 Calling tool: get_portfolio
🔧 Executing: get_portfolio()

✅ Tool Result:

{
  "success": true,
  "total_holdings": 6,
  "holdings": [
    {
      "isin": "INE270A01029",
      "cnc_used_quantity": 0,
      "collateral_type": "WC",
      "company_name": "ALOK INDUSTRIES LIMITED",
      "haircut": 0.2,
      "product": "D",
      "quantity": 250,
      "tradingsymbol": "ALOKINDS",
      "last_price": 15.0,
      "close_price": 15.0,
      "pnl": -825.0,
      "day_change": 0.0,
      "day_change_percentage": 0.0,
      "instrument_token": "NSE_EQ|INE270A01029",
      "average_price": 18.3,
      "collateral_quantity": 0,
      "collateral_update_quantity": 0,
      "trading_symbol": "ALOKINDS",
      "t1_quantity": 0,
      "exchange": "NSE"
    },
    {
      "isin": "INE040H01021",
      "cnc_used_quantity": 0,
      "collate

'{\n  "success": true,\n  "total_holdings": 6,\n  "holdings": [\n    {\n      "isin": "INE270A01029",\n      "cnc_used_quantity": 0,\n      "collateral_type": "WC",\n      "company_name": "ALOK INDUSTRIES LIMITED",\n      "haircut": 0.2,\n      "product": "D",\n      "quantity": 250,\n      "tradingsymbol": "ALOKINDS",\n      "last_price": 15.0,\n      "close_price": 15.0,\n      "pnl": -825.0,\n      "day_change": 0.0,\n      "day_change_percentage": 0.0,\n      "instrument_token": "NSE_EQ|INE270A01029",\n      "average_price": 18.3,\n      "collateral_quantity": 0,\n      "collateral_update_quantity": 0,\n      "trading_symbol": "ALOKINDS",\n      "t1_quantity": 0,\n      "exchange": "NSE"\n    },\n    {\n      "isin": "INE040H01021",\n      "cnc_used_quantity": 0,\n      "collateral_type": "WC",\n      "company_name": "SUZLON ENERGY LIMITED",\n      "haircut": 0.2,\n      "product": "D",\n      "quantity": 500,\n      "tradingsymbol": "SUZLON",\n      "last_price": 45.9,\n      "clo

In [51]:
# Example 5: Get orders
print("\n📊 Example 5: Check orders\n")
query_upstox_agent("Show me all my orders")


📊 Example 5: Check orders


🔍 User Query: Show me all my orders

🤖 Agent Processing...

📋 LLM Analysis...
Analysis: get_orders

📞 Calling tool: get_orders
🔧 Executing: get_orders()

✅ Tool Result:

{
  "success": false,
  "error": "400 Client Error: Bad Request for url: https://api.upstox.com/v2/order/orders"
}




'{\n  "success": false,\n  "error": "400 Client Error: Bad Request for url: https://api.upstox.com/v2/order/orders"\n}'

In [ ]:
# Example 6: Custom query
print("\n📊 Example 6: Custom query\n")
query_upstox_agent("What is my account margin and how much balance do I have available?")

In [35]:
# Debug: Test direct tool calls vs agent calls
print("🔍 Debugging Agent Tool Invocation\n")

# Test 1: Call the tool directly using .invoke()
print("="*70)
print("TEST 1: Direct Tool Call (Using .invoke())")
print("="*70)
try:
    result = search_instrument.invoke({"symbol": "RELIANCE"})
    print(f"Result: {json.dumps(result, indent=2)}\n")
except Exception as e:
    print(f"Error: {str(e)}\n")

# Test 2: Check agent state
print("="*70)
print("TEST 2: Agent Configuration")
print("="*70)
print(f"Agent type: {type(agent)}")
print(f"Tools registered: {len(tools)}")
for tool in tools:
    print(f"  - {tool.name}: {tool.description[:60]}...")

# Test 3: Try using agent.invoke with explicit messages
print("\n" + "="*70)
print("TEST 3: Agent Invoke with Simple Message")
print("="*70)
try:
    simple_query = "Search for RELIANCE stock"
    print(f"Query: {simple_query}\n")
    
    result = agent.invoke(
        {"messages": [HumanMessage(content=simple_query)]},
        config={"recursion_limit": 10}
    )
    
    print(f"Agent Result Keys: {result.keys()}")
    print(f"Messages: {len(result.get('messages', []))} messages")
    
    if result.get('messages'):
        last_msg = result['messages'][-1]
        print(f"\nLast message type: {type(last_msg).__name__}")
        print(f"Last message content:\n{last_msg.content[:300]}...")
        
except Exception as e:
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()


🔍 Debugging Agent Tool Invocation

TEST 1: Direct Tool Call (Using .invoke())
Result: {
  "success": true,
  "symbol": "RELIANCE",
  "type": "search",
  "message": "Symbol \"RELIANCE\" not found in portfolio. Common NSE symbols: RELIANCE, INFY, TCS, WIPRO, LT, MARUTI, HDFC, ICICIBANK, SBIN, BAJAJFINSV",
  "matches": 0,
  "instruments": []
}

TEST 2: Agent Configuration
Agent type: <class 'langgraph.graph.state.CompiledStateGraph'>
Tools registered: 7
  - get_instruments: Get list of user's portfolio holdings (available instruments...
  - search_instrument: Search for a specific instrument/stock by symbol (e.g., 'REL...
  - get_market_quote: Get real-time market quote for a specific instrument.
Return...
  - get_user_profile: Get authenticated user profile information.
Requires: ACCESS...
  - get_portfolio: Get user's current portfolio (holdings).
Requires: ACCESS_TO...
  - get_orders: Get list of all orders (placed, executed, cancelled).
Requir...
  - get_margins: Get account margin an

## How to Use This Agent

### 1. Set Up Credentials
Create a `.env` file with your Upstox credentials:
```
UPSTOX_API_KEY=your_key
UPSTOX_API_SECRET=your_secret
UPSTOX_ACCESS_TOKEN=your_token
```

### 2. Run the Notebook
Execute all cells from top to bottom to initialize the agent.

### 3. Query the Agent
Use natural language to query the Upstox API:

```python
query_upstox_agent("Search for TCS stock")
query_upstox_agent("Show me my portfolio")
query_upstox_agent("What are the current market quotes?")
query_upstox_agent("Get my account margins")
```

### Available Queries Examples
- "List all available instruments"
- "Search for INFY stock"
- "Get current market quote for RELIANCE"
- "Show my user profile"
- "What is my current portfolio?"
- "Show all my orders"
- "What are my account margins?"

## API Documentation
For more details, visit: https://upstox.com/developer/api-documentation/open-api
    

## Quick Reference - API Credential Steps

### 📱 In Your Browser:
1. Go to: **https://account.upstox.com/developer/apps**
2. **Create New App** → Fill app name & redirect URI
3. **Copy API Key** and **API Secret** from app details
4. **Click Generate** to create Access Token
5. **Copy the token** (valid for 1 year if generated manually)

### 📝 In Your Code Directory:
Create `.env` file:
```
UPSTOX_API_KEY=your_key_from_step_3
UPSTOX_API_SECRET=your_secret_from_step_3
UPSTOX_ACCESS_TOKEN=your_token_from_step_4
```

### ✅ Verify It Works:
Run the "Test Credentials" cell above to verify everything is working.

### 🔗 Using OAuth (Advanced)
If you need programmatic token generation:

**Step 1: Get Authorization Code**
```
https://api.upstox.com/v2/login/authorization/dialog?response_type=code&client_id=YOUR_API_KEY&redirect_uri=http://localhost:8000/callback&state=xyz
```

**Step 2: Exchange for Access Token**
```bash
curl -X POST 'https://api.upstox.com/v2/login/authorization/token' \
  -H 'Content-Type: application/x-www-form-urlencoded' \
  -d 'code=YOUR_AUTH_CODE&client_id=YOUR_API_KEY&client_secret=YOUR_API_SECRET&redirect_uri=http://localhost:8000/callback&grant_type=authorization_code'
```

### ❓ Troubleshooting
- **"API Key configured: False"** → Your `.env` file is missing or not loaded
- **"Access Token available: False"** → Generate a token in developer apps portal
- **"Invalid or expired ACCESS_TOKEN"** → Generate a new token
- **"No instruments found"** → API connection is working, symbol not found
